# Classificazione degli Esopianeti

Gli **esopianeti** sono pianeti che orbitano attorno a stelle diverse dal Sole. Dal 1995 ad oggi ne sono stati scoperti oltre 5000, grazie a missioni come **Kepler**, **TESS** e tecniche come il **transito** e la **velocità radiale**.

In questo notebook analizziamo il catalogo aperto degli esopianeti (`open_exoplanet_catalogue.txt`) per esplorare le caratteristiche dei pianeti confermati e delle loro stelle ospitanti.

## 1. Import delle librerie

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 2. Caricamento del catalogo

Il file `open_exoplanet_catalogue.txt` contiene i dati in formato TSV (tab-separated values). Verifichiamo il percorso prima di caricarlo.

In [ ]:
def carica_catalogo(percorso_file):
    """Carica il catalogo esopianeti da file TSV.

    Parametri:
        percorso_file: percorso del file .txt

    Restituisce:
        DataFrame pandas con i dati, oppure None in caso di errore.
    """
    print("Directory corrente:", os.getcwd())
    print("Il file esiste?", os.path.exists(percorso_file))

    try:
        df = pd.read_csv(percorso_file, sep='\t', on_bad_lines='skip', encoding='utf-8')
        print(f"\nCatalogo caricato: {len(df)} righe x {len(df.columns)} colonne")
        return df
    except pd.errors.ParserError as e:
        print(f"Errore durante il parsing: {e}")
        return None
    except FileNotFoundError:
        print(f"File non trovato: {percorso_file}")
        return None


file_path = './data/open_exoplanet_catalogue.txt'
df = carica_catalogo(file_path)

## 3. Esplorazione del DataFrame

Analizziamo la struttura, le statistiche descrittive e un campione dei dati.

In [ ]:
def esplora_dataframe(df):
    """Stampa informazioni riassuntive su un DataFrame."""
    if df is None:
        print("DataFrame non valido.")
        return

    print("=" * 66)
    print("COLONNE DISPONIBILI:")
    print(df.columns.tolist())
    print("=" * 66)

    print("\nINFORMAZIONI SUL DATAFRAME:")
    df.info()
    print("=" * 66)

    print("\nSTATISTICHE DESCRITTIVE (variabili numeriche):")
    print(df.describe())
    print("=" * 66)

    print("\nPRIME 5 RIGHE:")
    print(df.head())
    print("=" * 66)

esplora_dataframe(df)

## 4. Filtraggio: pianeti confermati

Selezioniamo solo i pianeti con stato "Confirmed planets" e con temperatura della stella ospitante nota.

In [ ]:
def filtra_pianeti_confermati(df):
    """Filtra il DataFrame per estrarre solo i pianeti confermati con temperatura nota."""
    colonne_interesse = ['name', 'hoststar_temperature', 'list']
    df_filtrato = df[colonne_interesse].copy()

    # Filtra per pianeti confermati
    df_filtrato = df_filtrato[df_filtrato['list'] == 'Confirmed planets']
    # Rimuove le righe senza temperatura
    df_filtrato = df_filtrato[df_filtrato['hoststar_temperature'].notna()]

    print(f"Pianeti confermati con temperatura nota: {len(df_filtrato)}")
    return df_filtrato


df_confermati = filtra_pianeti_confermati(df)
print(df_confermati)

## 5. Analisi della temperatura delle stelle ospitanti

Visualizziamo la temperatura superficiale delle stelle che ospitano pianeti confermati. Per chiarezza grafica, limitiamo ai primi 100 oggetti.

In [ ]:
def grafico_temperature_stelle(df, n_max=100):
    """Crea un grafico a barre delle temperature stellari.

    Parametri:
        df: DataFrame con colonne 'name' e 'hoststar_temperature'
        n_max: numero massimo di pianeti da mostrare
    """
    df_plot = df.head(n_max)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(df_plot['name'], df_plot['hoststar_temperature'], color='skyblue', edgecolor='navy')

    ax.set_xlabel('Pianeta', fontsize=12)
    ax.set_ylabel('Temperatura della stella ospitante (K)', fontsize=12)
    ax.set_title(f'Temperatura delle stelle ospitanti (primi {n_max} pianeti confermati)',
                 fontsize=14, fontweight='bold')
    ax.tick_params(axis='x', rotation=90)
    ax.grid(axis='y', linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()
    return fig


grafico_temperature_stelle(df_confermati, n_max=100)

## 6. Temperatura media

Calcoliamo la temperatura media delle stelle che ospitano pianeti confermati.

In [ ]:
def temperatura_media_stelle(df):
    """Calcola e stampa la temperatura media delle stelle ospitanti."""
    media = df['hoststar_temperature'].mean()
    print(f'Temperatura media delle stelle ospitanti: {media:.2f} K')
    return media


temp_media = temperatura_media_stelle(df_confermati)